# Task 2 EDA: Danish West Indies censuses 1846 and 1850

**Scope.** This notebook looks at the 1846 and 1850 census records of the Danish West Indies (DWI, FamilySearch partner data) and shows what the data contains: column types and gaps, how stable the fields are from 1846 to 1850, how noisy age is, what a birth-year blocking window costs, and how households look. It describes; **Phase 1 decides nothing** (blocking window width, feature weights and the household pass are left to the user afterwards, see the last section).

**Data governance.** The data is FamilySearch partner data. Every output below is an aggregate (counts, shares, distributions, plots). No record rows, no ids, no owner names. Name values appear only as frequency examples that occur at least 5 times. The committed notebook carries no outputs; the executed copy is reviewed locally first.

**Reading the numbers.** Descriptions of columns come from what the data shows, not from official definitions. Odd values are questions for the data owners, not errors fixed here.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

sys.path.insert(0, str(Path.cwd().resolve().parent / "scripts"))
import project_paths as pp

pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(40)
pl.Config.set_fmt_str_lengths(110)

DWI = pl.scan_parquet(pp.DWI_PARQUET)
CENSUSES = ("1846", "1850")
WINDOWS = (1, 2, 3, 5)
MIN_EXAMPLE_COUNT = 5
FACTS = {}

# Columns whose example values are never shown (identifiers and owner names).
MASK_EXAMPLE_COLUMNS = {
    "Column1",
    "pa_id",
    "inp_FS_RECORD_NBR",
    "inp_FS_HOUSEHOLD_ID",
    "inp_OWNER_NAME_GN_ORIG",
    "inp_OWNER_NAME_SURN_ORIG",
}


def filled(col):
    """Not null and not blank after strip. Accepts a column name or an expression."""
    e = pl.col(col) if isinstance(col, str) else col
    return e.is_not_null() & (e.str.strip_chars() != "")


def fact(key, value):
    """Record and print one computed fact as a FACT line."""
    if isinstance(value, float):
        value = round(value, 4)
    FACTS[key] = value
    print(f"FACT {key}={value}")


def print_facts(*prefixes):
    """Re-print already computed facts whose key starts with one of the prefixes."""
    for key in sorted(FACTS):
        if any(key.startswith(p) for p in prefixes):
            print(f"FACT {key}={FACTS[key]}")


def normalise_name(expr):
    """NFKD, drop combining marks, lowercase, punctuation to space, collapse whitespace, strip."""
    return (
        expr.cast(pl.String)
        .str.strip_chars()
        .str.normalize("NFKD")
        .str.replace_all(r"\p{Mn}", "")
        .str.to_lowercase()
        .str.replace_all(r"[^\w\s]", " ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


**Name normalisation** (used for anchors and agreement rates, kept simple on purpose): Unicode NFKD, remove combining marks, lowercase, replace punctuation by a space, collapse whitespace, strip. Letters such as the Danish "o with stroke" do not decompose and stay as they are. Null and blank strings are both treated as missing everywhere.

## 1. Columns, casts and parsing

All 22 source columns are strings in the parquet (nothing was coerced on conversion). Three columns are cast explicitly with `strict=False`, so a value that is filled but does not parse becomes null and is counted as a finding:

- `inp_PR_AGE` to `age`
- `inp_EVENT_YEAR` to `census_year`
- `inp_PR_BIR_YEAR_EST` to `bir_year_est`

`census` is the stripped `inp_EVENT_YEAR` string, `sex` the stripped `inp_PR_SEX_CODE`, and `status` the free/enslaved flag as `free`, `enslaved` or `blank`.

In [ ]:
D = (
    DWI.with_columns(
        pl.col("inp_EVENT_YEAR").str.strip_chars().alias("census"),
        pl.col("inp_PR_AGE").str.strip_chars().cast(pl.Int32, strict=False).alias("age"),
        pl.col("inp_EVENT_YEAR").str.strip_chars().cast(pl.Int32, strict=False).alias("census_year"),
        pl.col("inp_PR_BIR_YEAR_EST").str.strip_chars().cast(pl.Int32, strict=False).alias("bir_year_est"),
        pl.col("inp_PR_SEX_CODE").fill_null("").str.strip_chars().alias("sex"),
        pl.col("inp_PR_FLAG_FREE_OR_ENSLAVED_ORIG")
        .fill_null("")
        .str.strip_chars()
        .str.to_lowercase()
        .replace("", "blank")
        .alias("status"),
    )
    .collect()
)

cast_rows = []
for c in CENSUSES:
    sub = D.filter(pl.col("census") == c)
    fact(f"task2.rows.{c}", sub.height)
    for label, raw, parsed in [
        ("age", "inp_PR_AGE", "age"),
        ("census_year", "inp_EVENT_YEAR", "census_year"),
        ("bir_year_est", "inp_PR_BIR_YEAR_EST", "bir_year_est"),
    ]:
        is_filled = filled(raw)
        filled_n = sub.filter(is_filled).height
        fail_n = sub.filter(is_filled & pl.col(parsed).is_null()).height
        missing_n = sub.height - filled_n
        cast_rows.append({"cast": label, "census": c, "filled_n": filled_n, "parse_fail_n": fail_n, "missing_n": missing_n})
        if label == "age":
            fact(f"task2.age_parse_fail.{c}", fail_n)
            fact(f"task2.age_missing.{c}", missing_n)
        elif label == "bir_year_est":
            fact(f"task2.bir_year_parse_fail.{c}", fail_n)
            fact(f"task2.bir_year_missing.{c}", missing_n)
        else:
            fact(f"task2.census_year_parse_fail.{c}", fail_n)
pl.DataFrame(cast_rows)


**Is the birth-year estimate independent evidence?** PROJECT.md suspects that `inp_PR_BIR_YEAR_EST` is just the census year minus the age. The check below counts how often that holds among records where all three values parse. If the share is close to 1, the estimate adds nothing beyond age, and the blocking window further down is built on `census_year - age` instead.

In [ ]:
chk = D.filter(pl.col("age").is_not_null() & pl.col("census_year").is_not_null() & pl.col("bir_year_est").is_not_null())
chk = chk.with_columns((pl.col("bir_year_est") - (pl.col("census_year") - pl.col("age"))).alias("gap"))
fact("task2.biryear_is_derived_share", (chk["gap"] == 0).mean())
fact("task2.biryear_checked_n", chk.height)
chk.group_by("gap").len().rename({"len": "n"}).sort("n", descending=True).head(8)


### Column overview per census

One row per source column and census: a rough type profile, the share of filled values that parse as an integer, the share missing (null or blank), the number of distinct filled values, and up to three example values that occur at least 5 times, shown as `value (n)`. Identifier columns and owner-name columns are masked (`MASK_EXAMPLE_COLUMNS`); owner names are never shown. This describes what the data shows, not official column definitions.

In [ ]:
def column_overview(lf, census):
    sub = lf.filter(pl.col("inp_EVENT_YEAR").str.strip_chars() == census).collect()
    rows = []
    for col in sub.columns:
        s = sub.get_column(col)
        ok = s.filter(s.is_not_null() & (s.str.strip_chars() != ""))
        n_ok = ok.len()
        ints = ok.str.strip_chars().cast(pl.Int64, strict=False).is_not_null().sum() if n_ok else 0
        share = ints / n_ok if n_ok else None
        if n_ok == 0:
            profile = "empty"
        else:
            lens = ok.str.len_chars()
            span = f"{lens.min()}-{lens.max()}"
            if share == 1:
                profile = f"integer-like, {span} chars"
            elif share == 0:
                profile = f"text, {span} chars"
            else:
                profile = f"mixed integer/text, {span} chars"
        if col in MASK_EXAMPLE_COLUMNS:
            examples = "(not shown)"
        else:
            top = (
                sub.select(pl.col(col).alias("v"))
                .filter(filled("v"))
                .group_by("v")
                .len()
                .filter(pl.col("len") >= MIN_EXAMPLE_COUNT)
                .sort(["len", "v"], descending=[True, False])
                .head(3)
            )
            examples = "; ".join(f"{v} ({n})" for v, n in zip(top["v"], top["len"])) or "(none with n >= 5)"
        rows.append(
            {
                "column": col,
                "type_profile": profile,
                "int_parse_share": None if share is None else round(share, 3),
                "pct_missing": round(100 * (1 - n_ok / sub.height), 1),
                "n_unique": ok.n_unique(),
                "examples": examples,
            }
        )
    return pl.DataFrame(rows)


print("Column overview, census 1846")
column_overview(DWI, "1846")


In [ ]:
print("Column overview, census 1850")
column_overview(DWI, "1850")


### Value frequencies of code and place columns

These are low-cardinality code and place columns, not personal data. `(blank)` is a null or blank value.

In [ ]:
FREQ_COLUMNS = [
    "inp_PR_SEX_CODE",
    "inp_PR_FLAG_FREE_OR_ENSLAVED_ORIG",
    "inp_EVENT_TYPE",
    "inp_FS_RECORD_TYPE",
    "inp_PR_MARITAL_STATUS",
    "inp_EVENT_PLACE",
    "inp_EVENT_PLACE_SUP",
]
parts = []
for col in FREQ_COLUMNS:
    out = None
    for c in CENSUSES:
        t = (
            D.filter(pl.col("census") == c)
            .select(pl.col(col).fill_null("").str.strip_chars().alias("value"))
            .group_by("value")
            .len()
            .rename({"len": f"n_{c}"})
        )
        out = t if out is None else out.join(t, on="value", how="full", coalesce=True)
    out = out.fill_null(0).sort("value").with_columns(pl.lit(col).alias("column"))
    parts.append(out.select("column", "value", "n_1846", "n_1850"))
freq = pl.concat(parts).with_columns(pl.col("value").replace("", "(blank)"))
for c in CENSUSES:
    fact(f"task2.sex_blank.{c}", D.filter((pl.col("census") == c) & (pl.col("sex") == "")).height)
freq


### Island

The place columns are `inp_EVENT_PLACE` (town or island plus country) and `inp_EVENT_PLACE_SUP` (a variant spelling of the same place, blank for a few records). The frequency table above shows that `inp_EVENT_PLACE` is filled for every record and names the island (sometimes through a town such as Christiansted or Charlotte Amalie, sometimes spelled "Saint Thomas" or "St. Thomas"). So `island` is derived from **`inp_EVENT_PLACE`** with case-insensitive rules, applied in this order:

| island | rule on `inp_EVENT_PLACE` |
|---|---|
| St. Croix | contains `croix` |
| St. Thomas | contains `thomas` |
| St. John | contains the word `john` or the word `jan` |

Records that match no rule keep `island` null. They are counted (`task2.island_unmapped`) and kept in every later table as `(unmapped)`, never dropped.

In [ ]:
low = pl.col("inp_EVENT_PLACE").fill_null("").str.to_lowercase()
D = D.with_columns(
    pl.when(low.str.contains("croix")).then(pl.lit("St. Croix"))
    .when(low.str.contains("thomas")).then(pl.lit("St. Thomas"))
    .when(low.str.contains(r"\bjohn\b|\bjan\b")).then(pl.lit("St. John"))
    .otherwise(None)
    .alias("island")
)
fact("task2.island_unmapped", D.filter(pl.col("island").is_null()).height)
isl = (
    D.with_columns(pl.col("island").fill_null("(unmapped)"))
    .group_by("island", "census")
    .len()
    .pivot(on="census", index="island", values="len")
    .fill_null(0)
    .rename({"1846": "n_1846", "1850": "n_1850"})
    .select("island", "n_1846", "n_1850")
    .sort("island")
)
isl


## 2. Unique-name anchors and age noise

To measure how much age changes between the censuses without a ground truth, use **unique-name anchors**: records with a sex, a normalised given name and a normalised surname, whose key `sex|given|surname` (`name_key`) occurs **exactly once** in its census and is present in both censuses. Such a pair is very likely the same person. The anchor frame is kept in memory and never displayed; only aggregates are shown.

Anchors need a surname and a unique name, which makes them a biased subset (see the caveat below the plot).

In [ ]:
D = D.with_columns(
    normalise_name(pl.col("inp_PR_NAME_GN")).fill_null("").alias("gn_norm"),
    normalise_name(pl.col("inp_PR_NAME_SURN")).fill_null("").alias("sn_norm"),
).with_columns(
    (pl.col("sex") + "|" + pl.col("gn_norm") + "|" + pl.col("sn_norm")).alias("name_key")
)

ANCHOR_COLUMNS = {
    "age": "age",
    "status": "status",
    "island": "island",
    "inp_PR_BIR_PLACE_ORIG": "birthplace",
    "inp_OWNER_NAME_GN_ORIG": "owner_gn",
    "inp_OWNER_NAME_SURN_ORIG": "owner_sn",
    "inp_PR_NAME_SURN": "surname_raw",
}


def unique_keys(c):
    base = D.filter(
        (pl.col("census") == c) & (pl.col("sex") != "") & (pl.col("gn_norm") != "") & (pl.col("sn_norm") != "")
    )
    base = base.filter(pl.len().over("name_key") == 1)
    return base.select(
        "name_key", *[pl.col(src).alias(f"{dst}_{c}") for src, dst in ANCHOR_COLUMNS.items()]
    )


ANCHORS = unique_keys("1846").join(unique_keys("1850"), on="name_key", how="inner")
ANCHORS = ANCHORS.with_columns((pl.col("age_1850") - pl.col("age_1846")).alias("age_diff"))
fact("task2.anchors", ANCHORS.height)
fact("task2.anchor_free_share_1846", (ANCHORS["status_1846"] == "free").mean())
fact("task2.anchors_with_both_ages", ANCHORS["age_diff"].drop_nulls().len())
(
    ANCHORS.group_by("status_1846").len().rename({"len": "n"}).sort("n", descending=True)
)


In [ ]:
diff = ANCHORS["age_diff"].drop_nulls()
lo, hi = -10, 20
below, above = int((diff < lo).sum()), int((diff > hi).sum())
vc = diff.filter(diff.is_between(lo, hi)).value_counts()
counts = {int(k): int(v) for k, v in zip(vc[diff.name], vc["count"])}
xs = list(range(lo, hi + 1))
ys = [counts.get(x, 0) for x in xs]

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(xs, ys, color="#4c72b0")
ax.axvline(4, color="#c44e52", linestyle="--", linewidth=1, label="expected gap (1850 - 1846 = 4)")
ax.set_xlabel("age_1850 - age_1846 on anchors (clipped to -10..20)")
ax.set_ylabel("anchors")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print(f"Clipped tails: {below} anchors below {lo}, {above} anchors above {hi}")
fact("task2.agediff.median", float(diff.median()))
fact("task2.agediff.iqr", float(diff.quantile(0.75) - diff.quantile(0.25)))
for w in WINDOWS:
    fact(f"task2.agediff.within{w}", ((diff - 4).abs() <= w).mean())


**Caveat (anchor bias).** Anchors need a filled surname and a name that is unique within its census. Surnames are often empty for enslaved people, so anchors come mostly from free people. The spread of `age_1850 - age_1846` above is therefore a **lower bound** on age noise for enslaved people, who are less likely to have stable, recorded ages. The `within` facts are shares of anchors whose age difference deviates from the expected 4 years by at most that many years.

## 3. Window-cost table

Blocking on sex plus a birth-year window keeps the candidate set small enough to compare. Birth year is taken as `census_year - age` rather than `inp_PR_BIR_YEAR_EST`, because the check in section 1 shows the estimate is derived from age (so it is no independent evidence). Records without sex or without a parsed age cannot be blocked and are counted in `task2.blocking_excluded.<census>`.

Candidate pairs are computed **by aggregation** (per sex and birth year counts, joined and filtered); record pairs are never materialised. The sex-only baseline is the number of pairs with the same sex and no age restriction. `anchor_share` is the share of anchors (with both ages) that stay inside the window, so it is subject to the anchor-bias caveat above. The user picks the width afterwards; this table only shows the trade-off.

In [ ]:
B_ = D.filter((pl.col("sex") != "") & pl.col("age").is_not_null()).with_columns(
    (pl.col("census_year") - pl.col("age")).alias("birth_year")
)
for c in CENSUSES:
    total = D.filter(pl.col("census") == c).height
    kept = B_.filter(pl.col("census") == c).height
    fact(f"task2.blocking_excluded.{c}", total - kept)

cnt = B_.group_by("census", "sex", "birth_year").len().with_columns(pl.col("len").cast(pl.Int64).alias("n"))
c46 = cnt.filter(pl.col("census") == "1846").select("sex", pl.col("birth_year").alias("by_1846"), pl.col("n").alias("n_46"))
c50 = cnt.filter(pl.col("census") == "1850").select("sex", pl.col("birth_year").alias("by_1850"), pl.col("n").alias("n_50"))

sex_only = (
    c46.group_by("sex").agg(pl.col("n_46").sum())
    .join(c50.group_by("sex").agg(pl.col("n_50").sum()), on="sex")
    .select((pl.col("n_46") * pl.col("n_50")).sum())
    .item()
)
fact("task2.pairs.sex_only", sex_only)

pair_table = c46.join(c50, on="sex")
valid = ANCHORS["age_diff"].drop_nulls()
rows = [
    {"window": "sex only", "candidate_pairs_n": sex_only, "pairs_vs_sex_only_share": 1.0, "anchor_share": 1.0}
]
for w in WINDOWS:
    pairs = (
        pair_table.filter((pl.col("by_1846") - pl.col("by_1850")).abs() <= w)
        .select((pl.col("n_46") * pl.col("n_50")).sum())
        .item()
    )
    a_share = float(((valid - 4).abs() <= w).mean())
    fact(f"task2.window.{w}.pairs", pairs)
    fact(f"task2.window.{w}.anchor_share", a_share)
    rows.append(
        {
            "window": f"+-{w}",
            "candidate_pairs_n": pairs,
            "pairs_vs_sex_only_share": round(pairs / sex_only, 4),
            "anchor_share": round(a_share, 4),
        }
    )
pl.DataFrame(rows)


## Facts for open decisions

Every figure below is computed above and printed as a `FACT` line. Phase 1 makes none of these decisions.

In [ ]:
for key in sorted(FACTS):
    print(f"FACT {key}={FACTS[key]}")
